In [ ]:
# importing important libraries
import os
import shutil
import random
from pathlib import Path

In [ ]:
# setting the path to ldw dataset folder
LFW_ROOT = "lfw_dataset/lfw-deepfunneled/lfw-deepfunneled"
OUTPUT_ROOT = "dataset_split"
random.seed(42)

In [ ]:
# counting images for each class to segregate them
identity_counts = []

for person in os.listdir(LFW_ROOT):
    person_path = os.path.join(LFW_ROOT, person)

    if not os.path.isdir(person_path):
        continue

    images = [
        f

        for f in os.listdir(person_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    identity_counts.append((person, len(images)))
    
identity_counts.sort(key=lambda x: x[1], reverse=True)
print("Total identities:", len(identity_counts))

In [ ]:
# keeping identities/classes having atleast 10 images
eligible = [(person, count) for person, count in identity_counts if count >= 10]
print("Identities with >=10 images:", len(eligible))

In [ ]:
# selecting 130 classes
random.seed(42)
eligible_names = [person for person, count in eligible]
selected_identities = random.sample(eligible_names, 130)
print("Selected identities:", len(selected_identities))

In [ ]:
# train and test split (100 training classes and 30 testing classes)
random.seed(42)
random.shuffle(selected_identities)
train_identities = selected_identities[:100]
test_identities = selected_identities[100:]
print("Train identities:", len(train_identities))
print("Test identities:", len(test_identities))

In [ ]:
# creating training and testing folder
for split in ["train", "test"]:
    
    for subset in ["gallery", "probe"]:
        os.makedirs(os.path.join(OUTPUT_ROOT, split, subset), exist_ok=True)

In [ ]:
# creating gallery and probe folders in training dataset
def process_identity(person_name, split_name):
    src_dir = os.path.join(LFW_ROOT, person_name)

    images = [
        f for f in os.listdir(src_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)
    n = len(images)
    gallery_count = max(1, int(0.8 * n))
    gallery_images = images[:gallery_count]
    probe_images = images[gallery_count:]
    gallery_dir = os.path.join(OUTPUT_ROOT, split_name, "gallery", person_name)
    probe_dir = os.path.join(OUTPUT_ROOT, split_name, "probe", person_name)

    os.makedirs(gallery_dir, exist_ok=True)
    os.makedirs(probe_dir, exist_ok=True)

    for img in gallery_images:
        shutil.copy2(os.path.join(src_dir, img), os.path.join(gallery_dir, img))

    for img in probe_images:
        shutil.copy2(os.path.join(src_dir, img), os.path.join(probe_dir, img))
        
    return (len(gallery_images), len(probe_images))

In [ ]:
# creating gallery and probe split in training dataset

train_stats = []

for person in train_identities:
    g, p = process_identity(person, "train")
    train_stats.append((person, g, p))
print("Train split created.")

# creating gallery and probe split in testing dataset
test_stats = []

for person in test_identities:
    g, p = process_identity(person, "test")
    test_stats.append((person, g, p))
    
print("Test split created.")

In [ ]:
# verifying is split worked correctly or not
print("\nSample train identities:\n")

for row in train_stats[:10]:
    print(row)

print("\nSample test identities:\n")

for row in test_stats[:10]:
    print(row)